## Text Generation

In this notebook:

- Loading a pretrained GPT-2 model and tokenizer
- Tokenize a sentence and inspect the token IDs
- Look at the embedding vectors for each token
- Inspect real attention weights (Query-Key relevance scores) and verify causal masking
- Get logits and convert them to probabilities using softmax
- Compare small GPT-2 vs GPT-2-large predictions for the same sentence
- Build a manual text generation loop (predict → append → repeat)
- Hit and understand a real bug: greedy decoding causing an infinite loop
- Fix it using sampling instead of argmax

In [1]:
!pip install transformers -q

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### Tokenization
The text first goes through a **tokenizer**, which converts text into token IDs using a fixed vocabulary (GPT-2 has 50,257 possible tokens).

In [3]:
text = "The capital of France is"

input_ids = tokenizer.encode(text, return_tensors="pt")
print(input_ids)

tensor([[ 464, 3139,  286, 4881,  318]])


### Embeddings
Token IDs get looked up in `wte` (word token embeddings) : a learned
table inside the model that converts each ID into a vector (768 numbers for
GPT-2). This is where meaning starts getting represented. Different from the
tokenizer's vocabulary table, which is fixed and never learned.

In [4]:
# Get the embedding vector for each token ID
embeddings = model.transformer.wte(input_ids)

# Shape tells us: (1 sentence, 5 tokens, 768 numbers per token), 768 is GPT-2's embedding dimension
print(embeddings.shape)

torch.Size([1, 5, 768])


In [5]:
# attn_implementation="eager" tells the model to use the older,
# explicit attention computation — this is required if we want to actually see the attention weights, not just the final output
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [6]:
# output_attentions=True tells the model to also return the attention weights from every layer, not just the final prediction
# This runs the 5 tokens through the full model — embeddings → all the attention + MLP layers → logits
outputs = model(input_ids, output_attentions=True)

In [7]:
# outputs.attentions is a tuple with one entry per layer (GPT-2 has 12 layers, 12 heads)
# Each entry has shape: (batch, num_heads, tokens, tokens)
# Looking at layer 0, head 0 — the relevance scores between every pair of tokens
layer0_attention = outputs.attentions[0]
print(layer0_attention.shape)

torch.Size([1, 12, 5, 5])


In [8]:
# [0, 0] picks: sentence 0, head 0 — leaves us the 5x5 grid of scores
head0_scores = layer0_attention[0, 0]
print(head0_scores)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8464, 0.1536, 0.0000, 0.0000, 0.0000],
        [0.7014, 0.2237, 0.0749, 0.0000, 0.0000],
        [0.6077, 0.1788, 0.1439, 0.0696, 0.0000],
        [0.6099, 0.1519, 0.0626, 0.0945, 0.0812]], grad_fn=<SelectBackward0>)


### Logits → Probabilities
- After all 12 layers finish, every token has a "final
representation" — a 768-number vector, now updated by all that attention mixing.
- That final representation (vector of the last token) gets multiplied by the
UNEMBEDDING MATRIX. This matrix has one row for every possible token in the vocabulary (50257 rows total).
- This multiplication produces one score for every single possible token — one number per vocabulary word.

- These resulting scores are called LOGITS.
- Logits aren't probabilities yet — softmax converts them, so all values
  sum to 1.

In [9]:
# We only want the LAST token's row of logits (the one that predicts what word comes next), so we select index -1 (last position) here.
last_token_logits = outputs.logits[0, -1]

# The result is one number per vocabulary token (50257 numbers total)
print(last_token_logits.shape)

torch.Size([50257])


In [10]:
import torch

# Softmax converts raw scores (logits) into probabilities:
# torch.nn.functional.softmax does this math for us

probs = torch.nn.functional.softmax(last_token_logits, dim=-1)

print(probs.shape)
print(probs.sum())  # should print ~1.0, confirming it's a real probability distribution

torch.Size([50257])
tensor(1.0000, grad_fn=<SumBackward0>)


In [11]:
# Find the token ID with the highest probability
# torch.argmax returns the index of the largest value in the tensor.
top_token_id = torch.argmax(probs)

# Convert that token ID back into readable text using the tokenizer
predicted_word = tokenizer.decode(top_token_id)

print(predicted_word)

 the


In [12]:
# Get the top 5 highest-probability tokens instead of just the single best one
# torch.topk returns both the values (probabilities) and their indices (token IDs)
top5 = torch.topk(probs, 5)

for score, token_id in zip(top5.values, top5.indices):
    print(tokenizer.decode(token_id), "-", score.item())

 the - 0.08459090441465378
 now - 0.04794684424996376
 a - 0.046160366386175156
 France - 0.03237792104482651
 Paris - 0.03224455565214157


###Loading a bigger, more capable version of GPT-2

GPT-2 (small) did NOT confidently predict "Paris" for "The capital of
France is" : it was uncertain, spreading probability thin across
several plausible continuations (all under 10%). Paris was only 5th place.
Switching to `gpt2-large` to see if it improves.

In [13]:
# "gpt2-large" has significantly more parameters than base "gpt2",
# attn_implementation="eager" again, so we can still inspect attention if needed
tokenizer_large = GPT2Tokenizer.from_pretrained("gpt2-large")
model_large = GPT2LMHeadModel.from_pretrained("gpt2-large", attn_implementation="eager")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.25GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [14]:
# Tokenize the same sentence, but using the large model's tokenizer
input_ids_large = tokenizer_large.encode(text, return_tensors="pt")

# Run it through the bigger model
outputs_large = model_large(input_ids_large)

# Same steps as before: grab the last token's logits, convert to probabilities
last_token_logits_large = outputs_large.logits[0, -1]
probs_large = torch.nn.functional.softmax(last_token_logits_large, dim=-1)

# Top 5 predictions again
top5_large = torch.topk(probs_large, 5)
for score, token_id in zip(top5_large.values, top5_large.indices):
    print(tokenizer_large.decode(token_id), "-", score.item())

 the - 0.09703114628791809
 Paris - 0.07206082344055176
 a - 0.06962215155363083
 also - 0.040336694568395615
 located - 0.03953622281551361


Paris jumped from 5th to 2nd, and we can see more relevant words creeping into the list too (located, also) that weren't there with the smaller model. Generating sentences by appending generated words one by one:

In [15]:
# Start with the original sentence
current_ids = input_ids_large

# generating 10 new tokens, one at a time
for step in range(10):

    # Run the current sequence through the model
    outputs_step = model_large(current_ids)

    # Grab logits for the LAST token position only — that's the one predicting what comes next
    next_token_logits = outputs_step.logits[0, -1]

    # Convert to probabilities, then pick the single most likely token (this is "greedy" generation — always taking the top choice
    next_token_probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
    next_token_id = torch.argmax(next_token_probs).unsqueeze(0).unsqueeze(0)

    # Append the new token to our sequence
    current_ids = torch.cat([current_ids, next_token_id], dim=1)

    # Print the sentence as it grows, one token at a time
    print(tokenizer_large.decode(current_ids[0]))

The capital of France is the
The capital of France is the capital
The capital of France is the capital of
The capital of France is the capital of France
The capital of France is the capital of France.
The capital of France is the capital of France.

The capital of France is the capital of France.


The capital of France is the capital of France.

The
The capital of France is the capital of France.

The capital
The capital of France is the capital of France.

The capital of


### Greedy decoding breaks
Always picking the #1 token (greedy decoding) caused an infinite loop:
"The capital of France is the capital of France. The capital of France is..."
This happens because the model's own repeated context becomes the most
"confident" pattern to continue. This is a well-known failure mode :
it's why real systems use **sampling** (randomly picking based on
probability weight, not always the top choice) instead of pure greedy
decoding, to avoid getting stuck.

`argmax` (the greedy) ALWAYS picks the #1 highest token, no matter how small its lead is. Even if "the" was only 9% likely (barely ahead of "now" at 8%, "a" at 7%), argmax picks "the" with 100% certainty, every single time.


Sampling instead respects the REAL probability. If "the" is only 9% likely,
sampling picks "the" about 9% of the time, and picks something else
(weighted by ITS probability) the other 91% of the time.

`torch.multinomial` (sampling) picks a token randomly, weighted by its probability — so a token with 9% probability has a 9% chance of being picked.

In [16]:
# Same generation loop as before, but this time using Sampling

current_ids = input_ids_large

for step in range(10):
    outputs_step = model_large(current_ids)
    next_token_logits = outputs_step.logits[0, -1]

    # Optional: temperature controls how "random" the sampling is.
    # Lower temperature = more confident/predictable, closer to greedy
    # Higher temperature = more random/creative
    # We'll use 1.0 for now (no adjustment) to isolate just the sampling effect
    temperature = 1.0
    next_token_probs = torch.nn.functional.softmax(next_token_logits / temperature, dim=-1)

    # THIS is the actual fix: sample from the distribution instead of argmax
    next_token_id = torch.multinomial(next_token_probs, num_samples=1).unsqueeze(0)

    current_ids = torch.cat([current_ids, next_token_id], dim=1)
    print(tokenizer_large.decode(current_ids[0]))

The capital of France is none
The capital of France is none other
The capital of France is none other than
The capital of France is none other than Paris
The capital of France is none other than Paris France
The capital of France is none other than Paris France,
The capital of France is none other than Paris France, very
The capital of France is none other than Paris France, very famous
The capital of France is none other than Paris France, very famous for
The capital of France is none other than Paris France, very famous for Paris


## What I learned :)

- The flow of events: attention weights get computed and passed through
  multiple stacked layers (12, for GPT-2), and only at the very end does
  the unembedding matrix multiplication happen with the last embedding to produce logits — which then get converted into real probabilities using softmax.
- Got comfortable with the shapes at each step: token IDs, embeddings,
  attention weights, logits.
- Assumed greedy decoding (always picking the top prediction) would give the
  correct output- learned that it can actually trap the model in
  an infinite loop instead.
- Learned sampling as the fix: picking the next token randomly, weighted
  by its actual probability, instead of always forcing the single top choice.